<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 3 &nbsp;&middot;&nbsp; Optimisation and Training Challenges</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>How neural networks learn — gradient descent, the evolution of optimisers from SGD to Adam, learning rate control, common failure modes, regularisation techniques, and saving a trained model for production use.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| 3.1 Gradient Descent | Intuition · Vanilla, stochastic, mini-batch GD · Walking the loss surface |
| 3.2 Optimisers | The full evolution: SGD → Momentum → RMSprop → Adam · The math behind each step |
| 3.3 Learning Rate & Scheduling | The single most important hyperparameter · StepLR · CosineAnnealingLR |
| 3.4 Common Training Problems | Vanishing & exploding gradients · Weight initialisation · Gradient clipping |
| 3.5 Regularisation | L2 weight decay · Dropout · Batch Normalisation · Early stopping |
| 3.6 Activation Functions | Sigmoid · Tanh · ReLU · Leaky ReLU · How to choose per layer |
| 3.7 ModelPipeline | The reusable production wrapper used in every chapter from here on |

**Dataset:** UCI California Housing — loaded directly from scikit-learn (`fetch_california_housing`). No download or account required.

| Feature | Description |
|---------|-------------|
| `MedInc` | Median income in block group (in $10,000s) |
| `HouseAge` | Median house age in block group (years) |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block group population |
| `AveOccup` | Average number of household members |
| `Latitude` | Block group latitude |
| `Longitude` | Block group longitude |
| **Target** | **Median house value for block group (in $100,000s) — this is a regression task** |

> **This chapter introduces the `ModelPipeline`** — a production-ready wrapper that is used in every chapter from here on. By the end of Section 3.7 you will have a trained model saved to disk, loadable in any session, and ready for inference and retraining.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 3 — Setup
# No Kaggle token required — California Housing loads directly from scikit-learn.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet numpy pandas torch matplotlib scikit-learn dill

import os, copy, datetime, inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
print('Setup complete.')

---
## Data Loading and Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load and inspect the California Housing dataset
# ─────────────────────────────────────────────────────────────────────────────

raw = fetch_california_housing(as_frame=True)
df  = raw.frame   # pandas DataFrame: 8 feature columns + MedHouseVal target

print(f'Shape   : {df.shape}  ({df.shape[0]:,} housing block groups, {df.shape[1]} columns)')
print(f'Missing : {df.isnull().sum().sum()}  (none — this dataset is clean)')
print(f'Target  : MedHouseVal — range {df["MedHouseVal"].min():.2f} '
      f'to {df["MedHouseVal"].max():.2f}  (units: $100,000)')
print()
df.describe().round(2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing
#
# All 8 features are numeric — no categorical encoding needed.
#
# Why log-transform the target?
#   House values are right-skewed: a small number of very expensive properties
#   pulls the distribution far to the right. MSE loss applied to raw prices
#   over-penalises errors on expensive houses and under-penalises errors on
#   cheap ones. log1p(y) compresses the range so the model optimises evenly.
#   At inference, np.expm1() reverses the transform back to dollar values.
#
# Correct order of operations:
#   1. Separate features and target
#   2. Log-transform the target
#   3. Split BEFORE scaling  ← prevents data leakage
#   4. Fit StandardScaler on train only; transform all three splits
# ─────────────────────────────────────────────────────────────────────────────

feature_names = list(raw.feature_names)   # 8 names

X     = df[feature_names].to_numpy().astype('float32')
y_raw = df['MedHouseVal'].to_numpy().astype('float32')
y     = np.log1p(y_raw)   # log-transform target

# Train 70% / Val 15% / Test 15%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Fit scaler on training data only — transform all splits consistently
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

n_features = X_train.shape[1]

print('Preprocessing complete.')
print(f'  Features : {n_features}  (all numeric — StandardScaler applied)')
print(f'  Train    : {X_train.shape[0]:,} samples')
print(f'  Val      : {X_val.shape[0]:,} samples')
print(f'  Test     : {X_test.shape[0]:,} samples')
print(f'  Target   : log1p(MedHouseVal)  '
      f'mean={y_train.mean():.3f}  std={y_train.std():.3f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset and DataLoaders — reused in every section of this chapter
# ─────────────────────────────────────────────────────────────────────────────

class CaliforniaDataset(Dataset):
    """PyTorch Dataset wrapping pre-processed California Housing arrays."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # (N,) → (N, 1)
    def __len__(self):           return len(self.X)
    def __getitem__(self, idx):  return self.X[idx], self.y[idx]

train_dataset = CaliforniaDataset(X_train, y_train)
val_dataset   = CaliforniaDataset(X_val,   y_val)
test_dataset  = CaliforniaDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print('DataLoaders ready.')
print(f'  train_loader : {len(train_loader)} batches  ({len(train_dataset):,} samples, batch_size=64)')
print(f'  val_loader   : {len(val_loader)} batches')
print(f'  test_loader  : {len(test_loader)} batches')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Reusable training helpers — defined once, used throughout this chapter
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    """One full pass over the training data. Returns mean loss."""
    model.train()    # enables Dropout and BatchNorm training behaviour
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()                         # clear previous gradients
        predictions = model(X_batch)                  # forward pass
        loss = criterion(predictions, y_batch)        # compute loss
        loss.backward()                               # compute gradients
        optimiser.step()                              # update weights
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    """Evaluate on a DataLoader without updating weights. Returns mean loss."""
    model.eval()     # disables Dropout; BatchNorm uses running statistics
    total_loss = 0.0
    with torch.no_grad():    # no gradient tracking needed during evaluation
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total_loss += criterion(model(X_batch), y_batch).item() * len(X_batch)
    return total_loss / len(loader.dataset)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, verbose_every=10):
    """Train for num_epochs. Returns (train_losses, val_losses) histories.
    scheduler.step() is called after each epoch if provided.
    """
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:
            scheduler.step()
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
    return train_losses, val_losses


criterion = nn.MSELoss()
print('Helpers defined: train_one_epoch | evaluate | run_training')

---
# 3.1 Gradient Descent

## The core idea

Training a neural network means finding the values of all weights and biases that minimise the loss function. With potentially millions of parameters, this is a very high-dimensional optimisation problem. The algorithm that makes it tractable is **gradient descent**.

The intuition is simple: imagine you are standing on a hilly landscape in thick fog and want to reach the lowest point. You cannot see far ahead — but you can feel the slope under your feet. If you always take a small step in the direction that slopes downward most steeply, you will eventually reach a valley. Gradient descent does exactly this for the loss surface.

At every training step, for every weight $w$, gradient descent applies this update rule:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

**What does $\frac{\partial L}{\partial w}$ mean?**

The symbol $\frac{\partial L}{\partial w}$ is the **partial derivative** of the loss $L$ with respect to weight $w$. In plain terms, it answers the question: *if I increase $w$ by a tiny amount, how much does the loss change?*

- If $\frac{\partial L}{\partial w}$ is **positive** — increasing $w$ increases the loss. So we should *decrease* $w$.
- If $\frac{\partial L}{\partial w}$ is **negative** — increasing $w$ decreases the loss. So we should *increase* $w$.
- If $\frac{\partial L}{\partial w}$ is **large in magnitude** — $w$ has a strong influence on the loss right now.
- If $\frac{\partial L}{\partial w}$ is **close to zero** — $w$ has almost no effect, we are near a flat region or a minimum.

**Why subtract and not add?**

The gradient $\frac{\partial L}{\partial w}$ points in the direction that *increases* the loss — uphill. To reduce the loss we move in the *opposite* direction — hence the minus sign. The learning rate $\eta$ (eta) controls how large a step to take in that downhill direction.

| Symbol | Meaning |
|--------|--------|
| $w$ | A single weight being updated |
| $\frac{\partial L}{\partial w}$ | How much the loss changes when $w$ increases — computed by backpropagation |
| $\eta$ | Learning rate — the step size (a hyperparameter you set) |
| $\leftarrow$ | Assignment — the new value of $w$ after the update |


**Table 3.1 — Three variants of gradient descent**

| Variant | Data used per update | Gradient quality | Practical speed | Used in practice? |
|---------|---------------------|-----------------|----------------|------------------|
| **Vanilla (Batch) GD** | Entire dataset | Exact | Very slow — one update per full pass through all data | Rarely |
| **Stochastic GD (SGD)** | 1 sample | Very noisy | Fast — one update per sample | Rarely |
| **Mini-batch GD** | 32–256 samples | Good approximation | Fast and GPU-efficient | **Always** |

> **In practice, mini-batch GD is always used.** When practitioners say "SGD", they almost always mean mini-batch SGD. A batch size of 32 or 64 is a common starting point.
>
> **One weight update happens after each batch** — not after each sample (that would be stochastic GD) and not after the full dataset (that would be vanilla GD). So in one epoch with 14,447 training samples and a batch size of 64, the weights are updated approximately 226 times.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1a — Gradient descent from scratch on a 1-D problem
#
# We minimise  f(w) = (w - 3)^2  whose only minimum is at w = 3.
#
# Doing this by hand — before PyTorch handles it — makes the update rule
# concrete and removes the magic from the training loop later.
# ─────────────────────────────────────────────────────────────────────────────

def f(w):      return (w - 3.0) ** 2    # loss function
def grad_f(w): return 2.0 * (w - 3.0)  # exact gradient

w             = -2.0    # start far from optimum
learning_rate = 0.15
history       = []

print(f"{'Step':>4}  {'w':>8}  {'Loss':>10}  {'Gradient':>10}  {'Step taken':>12}")
print('-' * 56)

for step in range(20):
    loss      = f(w)
    gradient  = grad_f(w)
    step_size = -learning_rate * gradient
    history.append({'step': step, 'w': w, 'loss': loss})

    if step < 8 or step == 19:
        print(f"{step:>4}  {w:>8.4f}  {loss:>10.5f}  {gradient:>10.4f}  {step_size:>+12.4f}")
    elif step == 8:
        print('       ... converging ...')

    w = w - learning_rate * gradient    # the update rule

print()
print(f'Final w    = {w:.8f}   (true optimum = 3.0)')
print(f'Final loss = {f(w):.10f}')
print()
print('Key observation: the step size shrinks automatically at every iteration.')
print('As w approaches 3, the gradient approaches 0, so -lr * gradient approaches 0.')
print('No special logic is needed — this self-correction is built into the update rule.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1b — Visualise the descent path on the loss surface
# ─────────────────────────────────────────────────────────────────────────────

w_range    = np.linspace(-3, 7, 300)
loss_range = (w_range - 3.0) ** 2
steps_w    = [h['w']    for h in history]
steps_loss = [h['loss'] for h in history]

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.plot(w_range, loss_range, color='#2980b9', lw=2.5,
        label='Loss surface: f(w) = (w − 3)²')
ax.scatter(steps_w, steps_loss, color='#e74c3c', s=55, zorder=5, label='GD steps')
ax.plot(steps_w, steps_loss, color='#e74c3c', lw=1.2, linestyle='--', alpha=0.5)
ax.annotate('Start\nw = −2', xy=(steps_w[0], steps_loss[0]),
            xytext=(-1.5, 18), fontsize=9, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))
ax.annotate('Minimum\nw ≈ 3', xy=(3.0, 0.0), xytext=(4.5, 4),
            fontsize=9, color='#27ae60',
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.2))
ax.set_xlabel('Weight w', fontsize=11)
ax.set_ylabel('Loss f(w)', fontsize=11)
ax.set_title(
    'Figure 3.1 — Gradient Descent Walking the Loss Surface\n'
    'Steps shrink automatically as the gradient approaches zero near the minimum',
    fontsize=10, fontweight='bold', color='#1F3864')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 📝 Exercise 3.1 — Implement Gradient Descent

Minimise $f(w) = w^2 + 4w + 4$, which equals $(w + 2)^2$. The minimum is at $w = -2$, $f(-2) = 0$.

Run 20 steps starting from $w = 5$ for each of these learning rates: `0.05`, `0.5`, `1.1`.
Print the final $w$ and loss for each rate, then write a one-line comment explaining what you observe.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3.1
# ─────────────────────────────────────────────────────────────────────────────

def f_ex(w):      return w**2 + 4*w + 4    # = (w+2)^2, minimum at w = -2
def grad_f_ex(w): return 2*w + 4

for lr in [0.05, 0.5, 1.1]:
    w = 5.0
    # Your loop here (20 steps):

    print(f'lr={lr}: final w = ?   loss = ?')
    # Observation: ?

# See the Answer Key at the end of this chapter.

---
# 3.2 Optimisers

## The evolution: why each optimiser was invented

Each optimiser in this section was designed to fix a specific limitation of the one before it. Understanding the *problem each one solves* makes the choices intuitive rather than arbitrary.

---

### Step 1 — Vanilla SGD: where we start

Vanilla SGD applies the update rule from Section 3.1, once per mini-batch:

$$w \leftarrow w - \eta \cdot g_t$$

where $g_t$ is shorthand for $\frac{\partial L}{\partial w}$ at step $t$ — the gradient of the loss with respect to weight $w$, computed from the current batch. Every weight receives the same fixed step size $\eta$, regardless of how much it has been changing or in which direction it has been moving.

**Worked example — 5 steps on $f(w) = (w-3)^2$, starting at $w=0$, $\eta = 0.1$:**

| Step | $w$ | $g_t = 2(w-3)$ | Step $= -\eta \cdot g_t$ | New $w$ |
|------|------|---------------|--------------------------|--------|
| 1 | 0.0000 | −6.0000 | +0.6000 | 0.6000 |
| 2 | 0.6000 | −4.8000 | +0.4800 | 1.0800 |
| 3 | 1.0800 | −3.8400 | +0.3840 | 1.4640 |
| 4 | 1.4640 | −3.0720 | +0.3072 | 1.7712 |
| 5 | 1.7712 | −2.4576 | +0.2458 | 2.0170 |

After 5 steps: $w = 2.017$, still 0.98 away from the optimum $w = 3$. Each step is proportional to the gradient — as the gradient shrinks (because we are getting closer to 3), each step automatically gets smaller. **The same step size $\eta$ is applied to every parameter at every step regardless of history** — that is both the simplicity and the limitation of SGD.

This works — but has a well-known failure mode. Real loss surfaces have **ravines**: directions where the surface is nearly flat (small gradient, slow progress) and directions where it is very steep (large gradient, oscillating updates). SGD oscillates across the steep walls and crawls along the flat floor, making convergence slow and erratic.

---

### Step 2 — SGD with Momentum: adding memory

Momentum introduces a **velocity** $v_t$ that accumulates a running average of past gradients:

$$v_t \leftarrow \beta\, v_{t-1} + g_t \qquad w \leftarrow w - \eta\, v_t$$

**Reading the formula:** instead of updating $w$ directly with $g_t$ (today's gradient), we first update a velocity $v_t$ by blending the previous velocity $v_{t-1}$ with today's gradient $g_t$. The coefficient $\beta = 0.9$ means 90% of the old velocity is carried forward and 10% is the new gradient. Then we use this smoothed velocity to update $w$.

Think of a ball rolling downhill: in directions where gradients consistently point the same way (along the ravine floor), the velocity builds up over time — the ball accelerates. In directions where gradients alternate sign each step (across the steep walls), the positive and negative contributions cancel out — the oscillations damp.

**Why does $\beta = 0.9$ mean roughly the last 10 steps?**

Expanding the recurrence $v_t = \beta v_{t-1} + g_t$ backwards shows that $v_t$ is a weighted sum of all past gradients:

$$v_t = g_t + \beta\, g_{t-1} + \beta^2\, g_{t-2} + \beta^3\, g_{t-3} + \cdots$$

Each older gradient is multiplied by a higher power of $\beta$. With $\beta = 0.9$:

| Steps ago | Weight on that gradient |
|-----------|------------------------|
| 1 (most recent) | $0.9^0 = 1.000$ |
| 2 | $0.9^1 = 0.900$ |
| 5 | $0.9^4 = 0.656$ |
| 10 | $0.9^9 = 0.387$ |
| 20 | $0.9^{19} = 0.135$ |
| 30 | $0.9^{29} = 0.047$ |

A gradient from 10 steps ago still contributes with weight 0.387 — still meaningful. A gradient from 30 steps ago contributes with weight 0.047 — essentially negligible. So the "memory" is not a hard cutoff; it is a gradual fade. The rule of thumb $1 / (1 - \beta) = 1 / 0.1 = 10$ gives the approximate number of steps that carry significant weight — the **effective window size**.

**Remaining problem:** near the minimum, accumulated velocity carries the ball *past* the lowest point — it overshoots and keeps oscillating without settling.

**Worked example — 5 steps, same problem, $\eta = 0.1$, $\beta = 0.9$, starting $v_0 = 0$:**

| Step | $w$ | $g_t$ | $v_t = 0.9v_{t-1} + g_t$ | Step $= -\eta v_t$ | New $w$ |
|------|------|-------|--------------------------|-------------------|--------|
| 1 | 0.0000 | −6.0000 | −6.0000 | +0.6000 | 0.6000 |
| 2 | 0.6000 | −4.8000 | −10.2000 | +1.0200 | 1.6200 |
| 3 | 1.6200 | −2.7600 | −11.9400 | +1.1940 | 2.8140 |
| 4 | 2.8140 | −0.3720 | −11.1180 | +1.1118 | 3.9258 |
| 5 | 3.9258 | +1.8516 | −8.1546 | +0.8155 | 4.7413 |

After 5 steps: $w = 4.74$ — **momentum has overshot the optimum** ($w = 3$) and is still moving away from it. The velocity built up over steps 1–3 (reaching −11.94) carries the parameter past the minimum. By step 4 the gradient reverses sign (+1.85), but the velocity is still large and negative — so we keep moving in the wrong direction. This overshoot is the characteristic failure mode that Adam's adaptive learning rate corrects.

---

### Step 3 — RMSprop: per-parameter adaptive learning rate

RMSprop takes a different approach. Instead of tracking gradient direction, it tracks a **running average of the squared gradient** $s_t$ for each parameter separately:

$$s_t \leftarrow \rho\, s_{t-1} + (1 - \rho)\, g_t^2 \qquad w \leftarrow w - \frac{\eta}{\sqrt{s_t + \epsilon}}\, g_t$$

**Reading the formula:** $s_t$ is a running average of $g_t^2$ — the squared gradient. The update step for $w$ divides the gradient $g_t$ by $\sqrt{s_t}$ before scaling by $\eta$. The small constant $\epsilon$ (typically $10^{-8}$) prevents division by zero when $s_t$ is near zero.

**Why squaring?** Squaring makes $s_t$ always positive (direction doesn't matter, only magnitude) and gives larger weight to recent large gradients than to occasional small ones.

**The effect:** a parameter that consistently receives large gradients accumulates a large $s_t$ — the denominator $\sqrt{s_t}$ grows, so the effective step size $\eta / \sqrt{s_t}$ shrinks. A parameter with consistently small gradients has a small $s_t$ — the denominator stays small, so its effective step size is larger. This **adapts the learning rate per parameter automatically**.

**Remaining problem:** RMSprop has no momentum — it does not exploit consistent gradient directions, so convergence along flat ravine floors is still slow.

**Worked example — 5 steps, same problem, $\eta = 0.1$, $\rho = 0.9$, $\epsilon = 10^{-8}$, starting $s_0 = 0$:**

| Step | $w$ | $g_t$ | $g_t^2$ | $s_t = 0.9s + 0.1g^2$ | $\sqrt{s_t}$ | Eff. LR $= \eta/\sqrt{s}$ | Step | New $w$ |
|------|------|-------|---------|----------------------|------------|--------------------------|------|--------|
| 1 | 0.0000 | −6.0000 | 36.0000 | 3.6000 | 1.8974 | 0.05270 | +0.3162 | 0.3162 |
| 2 | 0.3162 | −5.3675 | 28.8105 | 6.1211 | 2.4741 | 0.04042 | +0.2170 | 0.5332 |
| 3 | 0.5332 | −4.9336 | 24.3408 | 7.9430 | 2.8183 | 0.03548 | +0.1751 | 0.7082 |
| 4 | 0.7082 | −4.5835 | 21.0088 | 9.2496 | 3.0413 | 0.03288 | +0.1507 | 0.8589 |
| 5 | 0.8589 | −4.2821 | 18.3365 | 10.1583 | 3.1872 | 0.03138 | +0.1344 | 0.9933 |

After 5 steps: $w = 0.993$. Two things to observe:
1. **The effective learning rate shrinks automatically** (0.0527 → 0.0314) as $s_t$ grows. This is adaptation in action — large gradients inflate $s_t$ and the denominator $\sqrt{s_t}$ absorbs the magnitude.
2. **The steps are steady but slow** — no overshoot (unlike Momentum), but also less progress per step. The gradient is still around −4 at step 5, but the effective step is only 0.13 because RMSprop is being conservative.

---

### Step 4 — Adam: the synthesis

Adam (Adaptive Moment Estimation) combines the two things the previous optimisers do separately:
- **SGD + Momentum** tracks *where the gradient has been pointing* — it builds velocity in consistent directions
- **RMSprop** tracks *how large the gradients have been* — it shrinks steps for parameters with large gradients

Adam does both simultaneously, for every parameter, with one additional fix that neither of them has.

#### What Adam tracks

Adam maintains two running averages per parameter at each step $t$:

**First moment $m_t$ — the momentum term:**
$$m_t \leftarrow \beta_1\, m_{t-1} + (1 - \beta_1)\, g_t$$

This is a weighted average of past gradients — same idea as Momentum. $\beta_1 = 0.9$ means 90% old average, 10% new gradient. It captures *direction* — which way has the gradient been pointing consistently?

**Second moment $v_t$ — the RMSprop term:**
$$v_t \leftarrow \beta_2\, v_{t-1} + (1 - \beta_2)\, g_t^2$$

This is a weighted average of squared past gradients — same idea as RMSprop. $\beta_2 = 0.999$ means 99.9% old average, 0.1% new squared gradient. It captures *magnitude* — how large have the gradients been recently? Because it is squared, it is always positive regardless of gradient sign.

#### The cold start problem and bias correction

Both $m_t$ and $v_t$ are initialised at zero. This creates a problem in the first few steps: the averages have barely built up any history, so they severely underestimate the true gradient signal.

**Concrete example at step $t=1$ with $g_1 = -6$:**
- $m_1 = 0.9 \times 0 + 0.1 \times (-6) = -0.6$ — but the true gradient is $-6$, not $-0.6$
- $v_1 = 0.999 \times 0 + 0.001 \times 36 = 0.036$ — but the true squared gradient is $36$, not $0.036$

Both are off by a factor of 10 or 1000 because we started from zero. If we used these raw values, the first steps would be far too small.

Adam corrects this by dividing by $(1 - \beta^t)$, which is small early on and approaches 1 as $t$ grows:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

At $t=1$: $\hat{m}_1 = -0.6\, /\, (1 - 0.9) = -0.6\, /\, 0.1 = -6.0$ — corrected back to the true gradient.  
At $t=1$: $\hat{v}_1 = 0.036\, /\, (1 - 0.999) = 0.036\, /\, 0.001 = 36.0$ — corrected back to the true squared gradient.  
By $t = 100$: $(1 - 0.9^{100}) \approx 1$, so $\hat{m}_{100} \approx m_{100}$ — the correction is essentially gone.

#### The update rule

$$w \leftarrow w - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}\, \hat{m}_t$$

Reading this in plain English: move in the direction of the smoothed gradient $\hat{m}_t$ (momentum), but scale the step size by $\eta / \sqrt{\hat{v}_t}$ — shrinking it for parameters where gradients have been large (RMSprop). The $\epsilon = 10^{-8}$ prevents division by zero.

**The key result:** because $\hat{m}_t$ is approximately the gradient and $\sqrt{\hat{v}_t}$ is approximately the gradient magnitude, their ratio $\hat{m}_t / \sqrt{\hat{v}_t}$ is approximately $\pm 1$ — the gradient normalised by its own size. This gives Adam nearly **constant-sized steps** regardless of whether gradients are large or small. That is why the Adam worked example below shows steps of almost exactly +0.100 at every iteration even though the raw gradient ranges from −6 to −5.

Default values: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, $\eta = 10^{-3}$. These work well across an extraordinarily wide range of problems — Adam is the default choice for nearly all tasks in this book.

**Worked example — 5 steps, same problem, $\eta = 0.1$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, starting $m_0 = v_0 = 0$:**

| Step | $w$ | $g_t$ | $m_t$ (raw) | $v_t$ (raw) | $\hat{m}_t$ (corrected) | $\hat{v}_t$ (corrected) | Eff. LR | Step | New $w$ |
|------|------|-------|------------|------------|------------------------|------------------------|---------|------|--------|
| 1 | 0.0000 | −6.0000 | −0.6000 | 0.036000 | **−6.0000** | **36.0000** | 0.01667 | +0.1000 | 0.1000 |
| 2 | 0.1000 | −5.8000 | −1.1200 | 0.069604 | −5.8947 | 34.8194 | 0.01695 | +0.0999 | 0.1999 |
| 3 | 0.1999 | −5.6002 | −1.5680 | 0.100897 | −5.7861 | 33.6659 | 0.01723 | +0.0997 | 0.2996 |
| 4 | 0.2996 | −5.4008 | −1.9513 | 0.129964 | −5.6740 | 32.5398 | 0.01753 | +0.0995 | 0.3991 |
| 5 | 0.3991 | −5.2018 | −2.2763 | 0.156893 | −5.5587 | 31.4414 | 0.01783 | +0.0991 | 0.4982 |

Notice at step 1: raw $m_1 = -0.6$ but bias-corrected $\hat{m}_1 = -6.0$ — a 10× inflation. Raw $v_1 = 0.036$ but bias-corrected $\hat{v}_1 = 36.0$ — a 1000× inflation. Without bias correction, the first step would be 100× smaller than it should be. After correction, the steps are +0.100 each — nearly constant, no overshoot, robust from step 1.

**Table 3.2a — Side-by-side comparison after 5 steps (all start at $w=0$, target $w=3$, $\eta=0.1$)**

| Step | SGD | SGD + Momentum | RMSprop | Adam |
|------|-----|---------------|---------|------|
| 1 | 0.6000 | 0.6000 | 0.3162 | 0.1000 |
| 2 | 1.0800 | 1.6200 | 0.5332 | 0.1999 |
| 3 | 1.4640 | 2.8140 | 0.7082 | 0.2996 |
| 4 | 1.7712 | **3.9258** ← past 3! | 0.8589 | 0.3991 |
| 5 | 2.0170 | **4.7413** ← moving away | 0.9933 | 0.4982 |

Each optimiser reveals its character clearly:
- **SGD:** steady, predictable progress — 2.02 after 5 steps
- **Momentum:** fastest early progress, but **overshoots** the target at step 4 and keeps accelerating away from it
- **RMSprop:** adaptive but conservative — effective LR shrinks as gradients stay large, reaching only 0.99
- **Adam:** most uniform steps (~0.100 each) due to bias correction — reaches 0.50 with no overshoot

---

**Table 3.2 — Optimiser summary**

| Optimiser | Problem it solves | Key mechanism | Default LR | Best used for |
|-----------|------------------|--------------|-----------|---------------|
| **SGD** | — (baseline) | Pure gradient step | 0.01–0.1 | Simple baselines; use with momentum |
| **SGD + Momentum** | Ravine oscillation | Velocity accumulation ($\beta=0.9$) | 0.01 | CNNs (Ch. 5); fine-tuning pre-trained models |
| **RMSprop** | Feature scale differences | Per-parameter adaptive LR | 1e-3 | RNNs / LSTMs (Ch. 6) |
| **Adam** | Both of the above | Momentum + adaptive LR + bias correction | 1e-3 | **Default for most tasks in this book** |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.2 — Baseline model used for optimiser comparison
# ─────────────────────────────────────────────────────────────────────────────

class BaselineNet(nn.Module):
    """Simple 3-layer feedforward network for regression.
    Used as the comparison model in Sections 3.2 and 3.3.
    """
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(),
            nn.Linear(128,   64), nn.ReLU(),
            nn.Linear(64,     1)
        )
    def forward(self, x): return self.net(x)

total_params = sum(p.numel() for p in BaselineNet(n_features).parameters())
print(f'BaselineNet: {n_features} → 128 → ReLU → 64 → ReLU → 1')
print(f'Total trainable parameters: {total_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.2 — Train with all four optimisers and compare convergence
#
# Each model is reset to the same random seed before training so differences
# in the loss curves reflect only the optimiser — not lucky initialisation.
# ─────────────────────────────────────────────────────────────────────────────

optimiser_configs = [
    ('SGD',            lambda p: optim.SGD(p,     lr=0.01)),
    ('SGD + Momentum', lambda p: optim.SGD(p,     lr=0.01, momentum=0.9)),
    ('RMSprop',        lambda p: optim.RMSprop(p, lr=1e-3)),
    ('Adam',           lambda p: optim.Adam(p,    lr=1e-3)),
]

opt_results = {}

for name, opt_fn in optimiser_configs:
    torch.manual_seed(42)
    model = BaselineNet(n_features).to(device)
    opt   = opt_fn(model.parameters())
    _, val_losses = run_training(
        model, train_loader, val_loader, criterion, opt,
        num_epochs=40, device=device, verbose_every=None
    )
    opt_results[name] = val_losses
    print(f'{name:20s}  final val loss: {val_losses[-1]:.4f}')

# Plot all four on one chart — the evolution story made visual
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

styles = [
    ('SGD',            '#bbbbbb', 1.5, '--'),
    ('SGD + Momentum', '#e67e22', 2.0, '-'),
    ('RMSprop',        '#9b59b6', 2.0, '-'),
    ('Adam',           '#27ae60', 2.5, '-'),
]
for (name, vals), (_, color, lw, ls) in zip(opt_results.items(), styles):
    ax.plot(vals, label=name, color=color, lw=lw, linestyle=ls)

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Validation MSE Loss', fontsize=11)
ax.set_title(
    'Figure 3.2 — The Evolution of Optimisers: SGD → Momentum → RMSprop → Adam\n'
    'Same model, same data, same initialisation — only the optimiser changes',
    fontsize=10, fontweight='bold', color='#1F3864')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print()
print('Each curve shows the direct benefit of the mechanism introduced in that step:')
print('  SGD → Momentum   : momentum reduces oscillation, steepens descent')
print('  Momentum → RMSprop: adaptive LR handles the feature scale differences in this dataset')
print('  RMSprop → Adam    : combining both gives the fastest, most stable convergence')

### 📝 Exercise 3.2 — Understand the Evolution

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3.2
# ─────────────────────────────────────────────────────────────────────────────

# Task 1 (conceptual): What specific failure mode does Momentum fix that
#   vanilla SGD cannot handle? Name it and explain in one sentence.
#   Answer: ?

# Task 2 (conceptual): RMSprop divides the gradient by sqrt(s_t) where
#   s_t is the running mean of squared gradients per parameter.
#   What happens to the effective step size of a parameter that consistently
#   receives very large gradients? Why is this desirable?
#   Answer: ?

# Task 3 (code): Train BaselineNet with AdamW for 40 epochs.
#   AdamW applies weight decay directly to the weights rather than
#   to the adaptive gradient estimate — often slightly better than Adam.
#   Compare its final val loss to Adam from the comparison above.

torch.manual_seed(42)
# model = ...
# opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
# ...

# See the Answer Key at the end of this chapter.

---
# 3.3 Learning Rate & Scheduling

The learning rate $\eta$ is the single hyperparameter with the greatest impact on training. It determines how large a weight update to make at each step. Getting it wrong is immediately visible:

**Table 3.3a — Effect of learning rate magnitude**

| Learning rate | What happens | What you see in the loss curve |
|--------------|-------------|-------------------------------|
| **Too large** | Steps overshoot the minimum — the model bounces across the valley | Loss oscillates, rises, or diverges to NaN |
| **Good** | Moves efficiently toward the minimum | Loss decreases smoothly and steadily |
| **Too small** | Takes tiny steps — barely moves | Loss decreases almost imperceptibly; training takes far too long |

## Learning rate scheduling

A higher learning rate in the early epochs moves quickly to the general region of the minimum. A lower learning rate in later epochs settles precisely into it. **Scheduling** automates this reduction.

**Table 3.3b — Common PyTorch schedulers**

| Scheduler | Behaviour | Key parameters | When to use |
|-----------|-----------|---------------|-------------|
| `StepLR` | Multiplies LR by `gamma` every `step_size` epochs | `step_size=15, gamma=0.5` | Simple, predictable decay; good default |
| `CosineAnnealingLR` | Smoothly reduces LR along a cosine curve to near-zero | `T_max=num_epochs` | Longer runs; often achieves better final loss |
| `ReduceLROnPlateau` | Reduces LR by `factor` when val loss has not improved for `patience` epochs | `patience=5, factor=0.5` | When training duration is uncertain |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.3a — Effect of learning rate: too large / good / too small
# ─────────────────────────────────────────────────────────────────────────────

lr_experiments = [
    ('lr = 0.1   (too large)', 0.1),
    ('lr = 1e-3  (good)',      1e-3),
    ('lr = 1e-6  (too small)', 1e-6),
]

lr_results = {}
for label, lr_val in lr_experiments:
    torch.manual_seed(42)
    model = BaselineNet(n_features).to(device)
    opt   = optim.Adam(model.parameters(), lr=lr_val)
    _, vl = run_training(model, train_loader, val_loader, criterion, opt,
                         num_epochs=40, device=device, verbose_every=None)
    lr_results[label] = vl
    print(f'{label:30s}  final val loss: {vl[-1]:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
for (label, vals), color in zip(lr_results.items(), ['#e74c3c', '#27ae60', '#aaaaaa']):
    ax.plot(vals, label=label, color=color, lw=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Validation MSE Loss', fontsize=11)
ax.set_title('Figure 3.3a — Effect of Learning Rate Magnitude',
             fontsize=10, fontweight='bold', color='#1F3864')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.3b — Learning rate scheduling: StepLR vs CosineAnnealingLR
#
# Both start from a higher LR (1e-2) and decay it over training.
# We compare to a fixed lr=1e-3 baseline (no scheduler).
# ─────────────────────────────────────────────────────────────────────────────

NUM_EPOCHS  = 60
sched_results = {}

# Baseline: fixed lr=1e-3
torch.manual_seed(42)
model = BaselineNet(n_features).to(device)
_, vl = run_training(model, train_loader, val_loader, criterion,
                     optim.Adam(model.parameters(), lr=1e-3),
                     num_epochs=NUM_EPOCHS, device=device, verbose_every=None)
sched_results['Fixed lr=1e-3  (no scheduler)'] = vl

# StepLR: halve the LR every 15 epochs
torch.manual_seed(42)
model = BaselineNet(n_features).to(device)
opt   = optim.Adam(model.parameters(), lr=1e-2)
sched = optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)
_, vl = run_training(model, train_loader, val_loader, criterion, opt,
                     num_epochs=NUM_EPOCHS, device=device,
                     scheduler=sched, verbose_every=None)
sched_results['StepLR  lr=1e-2 → halves every 15 epochs'] = vl

# CosineAnnealingLR: smooth cosine decay from lr=1e-2 to ~0
torch.manual_seed(42)
model = BaselineNet(n_features).to(device)
opt   = optim.Adam(model.parameters(), lr=1e-2)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
_, vl = run_training(model, train_loader, val_loader, criterion, opt,
                     num_epochs=NUM_EPOCHS, device=device,
                     scheduler=sched, verbose_every=None)
sched_results['CosineAnnealingLR  lr=1e-2 → ~0'] = vl

print('Final validation losses:')
for label, vals in sched_results.items():
    print(f'  {label:48s}: {vals[-1]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
for (label, vals), color, ls in zip(sched_results.items(),
                                     ['#aaaaaa', '#e67e22', '#2980b9'],
                                     ['--', '-', '-']):
    ax.plot(vals, label=label, color=color, lw=2, linestyle=ls)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Validation MSE Loss', fontsize=11)
ax.set_title('Figure 3.3b — LR Scheduling vs Fixed Learning Rate',
             fontsize=10, fontweight='bold', color='#1F3864')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 📝 Exercise 3.3 — Learning Rate Scheduling

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3.3
# ─────────────────────────────────────────────────────────────────────────────

# Task: Train BaselineNet with Adam (lr=1e-3) and ReduceLROnPlateau
#       (patience=5, factor=0.5) for 60 epochs.
#
#       Important: ReduceLROnPlateau.step() takes the val loss as an argument
#       and is called AFTER evaluate(), not inside run_training().
#       You will need to write the epoch loop manually.
#
#       Print the learning rate at epochs 1, 20, 40, 60.
#       Compare the final val loss to StepLR and CosineAnnealing above.

torch.manual_seed(42)
# model = ...
# opt   = ...
# sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
# for epoch in range(60):
#     ...
#     sched.step(val_loss)   # <-- note: takes val_loss, not just step()

# See the Answer Key at the end of this chapter.

---
# 3.4 Common Training Problems

## Vanishing and Exploding Gradients

During backpropagation, the gradient for each layer is computed by applying the chain rule — multiplying gradients from every subsequent layer together. In deep networks this chain of multiplications can go wrong in two opposite ways:

**Table 3.4a — Gradient problems: causes, symptoms, and fixes**

| Problem | What happens | Symptom | Fixes |
|---------|-------------|---------|-------|
| **Vanishing gradients** | Gradients shrink exponentially toward zero as they travel backward through layers | Early layers receive near-zero updates; training stalls; loss plateaus very early | ReLU activations · He initialisation · Batch Normalisation |
| **Exploding gradients** | Gradients grow exponentially as they travel backward | Loss becomes `NaN` or `Inf`; weights diverge immediately | Gradient clipping · Better initialisation · Lower learning rate |

## Weight Initialisation

How weights are set before training begins has a lasting impact:

- **All zeros:** every neuron in a layer computes the same output and receives the same gradient — they are permanently identical. The network never differentiates. Training fails completely.
- **Too large:** activations saturate (Sigmoid, Tanh) or gradient norms explode from the very first step.
- **Principled initialisation:** scales initial weights based on the number of inputs to each layer, so signal variance remains stable as it propagates forward and gradients remain stable as they propagate backward.

**Table 3.4b — Weight initialisation methods**

| Method | Std of initial weights | Design goal | Use with |
|--------|----------------------|-------------|----------|
| **Xavier / Glorot** | $\sqrt{2\,/\,(n_{in} + n_{out})}$ | Keep signal variance equal forward and backward | Sigmoid, Tanh |
| **He / Kaiming** | $\sqrt{2\,/\,n_{in}}$ | Compensate for ReLU zeroing half its inputs | **ReLU variants — PyTorch default for `nn.Linear`** |
| **All zeros / constant** | 0 | — | **Never use for weights** |

**Why do the formulas depend on $n_{in}$ and $n_{out}$?**

When a neuron computes $z = w_1 x_1 + w_2 x_2 + \cdots + w_{n} x_n$, the variance of $z$ is $n_{in} \cdot \text{Var}(w) \cdot \text{Var}(x)$. If $\text{Var}(w)$ is too large, $z$ grows exponentially through layers. If too small, $z$ shrinks to zero. Both formulas choose $\text{Var}(w)$ so that the variance of activations stays roughly constant across layers — the signal neither explodes nor vanishes.

**Why is He different from Xavier?**

Xavier was derived assuming linear activations — where the full signal passes through. ReLU zeros out all negative inputs, which effectively halves the signal strength at each layer. He init doubles the variance (note the $\sqrt{2/n_{in}}$ vs Xavier's $\sqrt{2/(n_{in}+n_{out})}$) to compensate for this half-zeroing, keeping signal variance stable specifically for ReLU networks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.4a — Demonstrate exploding gradients and fix with clip_grad_norm_
#
# A deliberately deep network with very large initial weights (std=5)
# produces enormous gradient norms from step one. clip_grad_norm_ bounds
# the total gradient norm while preserving direction.
# ─────────────────────────────────────────────────────────────────────────────

class DeepNet(nn.Module):
    """Six-layer network — depth amplifies gradient problems, making them easy to observe."""
    def __init__(self, n_in):
        super().__init__()
        dims   = [n_in, 64, 64, 64, 64, 64, 1]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


def apply_bad_init(model, std=5.0):
    """Replace all weights with N(0, std²) — intentionally too large."""
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, mean=0.0, std=std)
            nn.init.zeros_(layer.bias)
    return model


def grad_norm(model):
    """Total L2 norm of all gradients — a single number summarising gradient magnitude."""
    return sum(
        p.grad.data.norm(2).item() ** 2
        for p in model.parameters() if p.grad is not None
    ) ** 0.5


X_b, y_b = next(iter(train_loader))
X_b, y_b = X_b.to(device), y_b.to(device)

print('Comparing gradient norms over 5 batches:')
print(f"{'Batch':>5}  {'No clipping':>30}  {'With clip_grad_norm_(1.0)':>32}")
print('-' * 72)

for batch_idx in range(5):
    # Without clipping
    torch.manual_seed(0)
    m_bad = apply_bad_init(DeepNet(n_features), std=5.0).to(device)
    o_bad = optim.SGD(m_bad.parameters(), lr=0.01)
    o_bad.zero_grad()
    loss_bad = criterion(m_bad(X_b), y_b)
    loss_bad.backward()
    norm_bad = grad_norm(m_bad)

    # With clipping
    torch.manual_seed(0)
    m_clip = apply_bad_init(DeepNet(n_features), std=5.0).to(device)
    o_clip = optim.SGD(m_clip.parameters(), lr=0.01)
    o_clip.zero_grad()
    loss_clip = criterion(m_clip(X_b), y_b)
    loss_clip.backward()
    torch.nn.utils.clip_grad_norm_(m_clip.parameters(), max_norm=1.0)
    norm_clip = grad_norm(m_clip)

    print(f'{batch_idx:>5}  loss={loss_bad.item():>7.2f}  grad_norm={norm_bad:>10.2f}  '
          f'loss={loss_clip.item():>7.2f}  grad_norm={norm_clip:>6.4f}  (≤1.0)')

print()
print('clip_grad_norm_ scales ALL gradients down proportionally when their total')
print('norm exceeds max_norm. Direction is preserved — only magnitude is capped.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.4b — He initialisation (PyTorch default) vs bad initialisation
# ─────────────────────────────────────────────────────────────────────────────

init_results = {}

# He init — PyTorch applies this automatically to nn.Linear
torch.manual_seed(42)
model_he = DeepNet(n_features).to(device)   # default init = He
_, vl_he = run_training(model_he, train_loader, val_loader, criterion,
                         optim.Adam(model_he.parameters(), lr=1e-3),
                         num_epochs=30, device=device, verbose_every=None)
init_results['He init (PyTorch default)'] = vl_he

# Bad init with gradient clipping to prevent immediate divergence
torch.manual_seed(42)
model_bad = apply_bad_init(DeepNet(n_features), std=5.0).to(device)
opt_bad   = optim.Adam(model_bad.parameters(), lr=1e-3)
vl_bad = []
for _ in range(30):
    model_bad.train()
    for X_b2, y_b2 in train_loader:
        X_b2, y_b2 = X_b2.to(device), y_b2.to(device)
        opt_bad.zero_grad()
        criterion(model_bad(X_b2), y_b2).backward()
        torch.nn.utils.clip_grad_norm_(model_bad.parameters(), max_norm=1.0)
        opt_bad.step()
    vl_bad.append(evaluate(model_bad, val_loader, criterion, device))
init_results['Bad init std=5 + gradient clipping'] = vl_bad

print('Final validation losses after 30 epochs:')
for label, vals in init_results.items():
    print(f'  {label:40s}: {vals[-1]:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
for (label, vals), color in zip(init_results.items(), ['#27ae60', '#e74c3c']):
    ax.plot(vals, label=label, color=color, lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE Loss')
ax.set_title('Figure 3.4 — He Initialisation vs Bad Initialisation (both with Adam)',
             fontsize=10, fontweight='bold', color='#1F3864')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 📝 Exercise 3.4 — Weight Initialisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3.4
# ─────────────────────────────────────────────────────────────────────────────

# Task 1: Write an apply_xavier_init() function using nn.init.xavier_uniform_
#         Train DeepNet with Xavier init for 30 epochs and print final val loss.

# Task 2: Xavier is designed for Sigmoid/Tanh, not ReLU. Does it still
#         converge on this dataset? Explain why He init is still preferred
#         for ReLU networks even if Xavier appears to work.
#   Answer: ?

# See the Answer Key at the end of this chapter.

---
# 3.5 Regularisation

**Regularisation** is any technique that reduces overfitting — the gap between how well a model performs on training data versus new, unseen data.

Overfitting happens when the model memorises training examples rather than learning the underlying pattern. It is recognisable by one reliable sign: **training loss keeps falling while validation loss flattens or rises**.

**Table 3.5 — Regularisation techniques**

| Technique | How it works | Applied where | PyTorch usage |
|-----------|-------------|--------------|---------------|
| **L2 weight decay** | Adds $\lambda \sum w^2$ penalty — pushes weights toward zero, preventing any single weight from dominating | Optimiser | `optim.Adam(params, weight_decay=1e-4)` |
| **Dropout** | Randomly zeros a fraction `p` of neuron outputs at each training step — forces the network to build redundant representations | After activation layers | `nn.Dropout(p=0.3)` |
| **Batch Normalisation** | Normalises each layer's pre-activation outputs to zero mean and unit variance per mini-batch — stabilises training and provides a mild regularisation effect | After linear, before activation | `nn.BatchNorm1d(num_features)` |
| **Early stopping** | Monitors val loss after each epoch and stops when it has not improved for `patience` epochs — prevents over-training | Training loop | Track best val loss; restore best weights |

> **Correct layer ordering:** `Linear → BatchNorm → Activation → Dropout`  
> BatchNorm normalises the raw linear output before the non-linearity; Dropout is applied after activation so it zeroes already-activated values, not pre-activation sums.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.5a — Regularised network: BatchNorm + Dropout + L2 weight decay
# ─────────────────────────────────────────────────────────────────────────────

class RegularisedNet(nn.Module):
    """Network with BatchNorm and Dropout in the correct order.

    Layer order: Linear → BatchNorm → ReLU → Dropout
    L2 weight decay is applied via the optimiser's weight_decay argument.
    """
    def __init__(self, n_in, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128),
            nn.BatchNorm1d(128),     # normalise across the batch
            nn.ReLU(),
            nn.Dropout(dropout_p),   # zero out dropout_p fraction of activations

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)


NUM_EPOCHS   = 80
reg_results  = {}

for label, model_fn, wd in [
    ('Baseline — no regularisation',        lambda: BaselineNet(n_features),       0),
    ('Dropout(0.3) + BatchNorm + L2(1e-4)', lambda: RegularisedNet(n_features, 0.3), 1e-4),
]:
    torch.manual_seed(42)
    model = model_fn().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=wd)
    tl_h, vl_h = run_training(model, train_loader, val_loader, criterion, opt,
                               num_epochs=NUM_EPOCHS, device=device, verbose_every=None)
    reg_results[label] = {'train': tl_h, 'val': vl_h}
    gap = vl_h[-1] - tl_h[-1]
    print(f'{label}')
    print(f'  train={tl_h[-1]:.4f}   val={vl_h[-1]:.4f}   train/val gap={gap:.4f}')
    print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')
for ax, (label, res), color in zip(axes, reg_results.items(), ['#e74c3c', '#27ae60']):
    ax.set_facecolor('white')
    ax.plot(res['train'], label='Train', color=color, lw=2)
    ax.plot(res['val'],   label='Val',   color=color, lw=2, linestyle='--')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9)
fig.suptitle('Figure 3.5 — Regularisation Closes the Train/Val Gap',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.5b — Early Stopping
#
# Rather than choosing a fixed number of epochs, early stopping monitors
# val loss after every epoch and stops when improvement ceases.
# The model is automatically reset to the weights that achieved the best
# val loss — not the weights at the final (potentially overfit) epoch.
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopping:
    """Halt training when validation loss stops improving.

    Args:
        patience  : consecutive epochs with no improvement before stopping
        min_delta : minimum change that counts as an improvement
    """
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None

    def step(self, val_loss, model):
        """Call after each epoch. Returns True if training should stop."""
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        """Load the best-seen weights back into the model."""
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


torch.manual_seed(42)
model_es = RegularisedNet(n_features, dropout_p=0.3).to(device)
opt_es   = optim.Adam(model_es.parameters(), lr=1e-3, weight_decay=1e-4)
es       = EarlyStopping(patience=8)

print('Training with early stopping (patience=8, up to 200 epochs):')
print(f"{'Epoch':>6}  {'Train':>8}  {'Val':>8}  {'Counter':>8}")
print('-' * 38)

for epoch in range(200):
    tl = train_one_epoch(model_es, train_loader, criterion, opt_es, device)
    vl = evaluate(model_es, val_loader, criterion, device)
    if epoch % 10 == 0:
        print(f'{epoch:>6}  {tl:>8.4f}  {vl:>8.4f}  {es.counter:>8}')
    if es.step(vl, model_es):
        print(f'\nEarly stopping at epoch {epoch}  '
              f'(best val loss: {es.best_loss:.4f})')
        es.restore_best(model_es)
        print('Best weights restored.')
        break

final_val = evaluate(model_es, val_loader, criterion, device)
print(f'\nFinal val loss after weight restore: {final_val:.4f}')

### 📝 Exercise 3.5 — Regularisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 3.5
# ─────────────────────────────────────────────────────────────────────────────

# Task 1: Train RegularisedNet with dropout_p in [0.1, 0.3, 0.5] for 60 epochs.
#         Print final train loss, val loss, and train/val gap for each.

# Task 2: You will notice that the train loss for the Dropout model is HIGHER
#         than the baseline, even though val loss is lower.
#         Explain why this happens and why it is not a problem.
#   Answer: ?

# See the Answer Key at the end of this chapter.

---
# 3.6 Activation Functions

Without activation functions, a stack of linear layers — no matter how deep — is mathematically equivalent to a single linear layer. Activation functions introduce **non-linearity**, enabling networks to approximate complex patterns.

**Table 3.6 — Activation function reference**

| Function | Formula | Output range | Properties | Where to use |
|----------|---------|-------------|-----------|-------------|
| **Sigmoid** | $1\,/\,(1+e^{-x})$ | (0, 1) | Smooth; saturates for large \|x\| causing vanishing gradients | Output layer: binary classification |
| **Tanh** | $(e^x - e^{-x})\,/\,(e^x + e^{-x})$ | (−1, 1) | Zero-centred — better than Sigmoid; still saturates | RNN hidden states (Ch. 6) |
| **ReLU** | $\max(0,\, x)$ | [0, ∞) | No vanishing gradient for $x>0$; fast; can produce dead neurons when $x<0$ always | **Default for all hidden layers** |
| **Leaky ReLU** | $x$ if $x>0$; else $\alpha x$ | (−∞, ∞) | Fixes dead neurons with a small slope $\alpha=0.01$ for negative inputs | When dead ReLU neurons are suspected |
| **Softmax** | $e^{x_i}\,/\,\sum_j e^{x_j}$ | (0, 1), sums to 1 | Converts raw scores to a probability distribution over classes | Output layer: multi-class classification |

> **Dead neurons:** A ReLU neuron is "dead" if it receives a negative pre-activation for every sample in every batch — it permanently outputs zero and receives no gradient. This can happen with large learning rates or bad initialisation. Leaky ReLU prevents this by allowing a small negative gradient ($\alpha = 0.01$) to flow even when $x < 0$.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.6a — Visualise the four main activation functions
# ─────────────────────────────────────────────────────────────────────────────

x = np.linspace(-4, 4, 300)

activations = {
    'Sigmoid':    (1 / (1 + np.exp(-x)),          (-0.15, 1.15)),
    'Tanh':       (np.tanh(x),                     (-1.25, 1.25)),
    'ReLU':       (np.maximum(0, x),               (-0.4,  4.2)),
    'Leaky ReLU': (np.where(x > 0, x, 0.01 * x),  (-0.45, 4.2)),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.patch.set_facecolor('white')
colors = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']

for ax, (name, (vals, ylim)), color in zip(axes, activations.items(), colors):
    ax.set_facecolor('white')
    ax.plot(x, vals, color=color, lw=2.5)
    ax.axhline(0, color='#cccccc', lw=0.8, linestyle='--')
    ax.axvline(0, color='#cccccc', lw=0.8, linestyle='--')
    ax.set_title(name, fontsize=11, fontweight='bold', color='#1F3864')
    ax.set_xlabel('Input x', fontsize=10)
    ax.set_ylim(*ylim)
    ax.set_xlim(-4, 4)

axes[0].set_ylabel('Output', fontsize=10)
fig.suptitle('Figure 3.6 — Activation Functions',
             fontsize=12, fontweight='bold', color='#1F3864', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.6b — ReLU vs Sigmoid in hidden layers: training comparison
#
# Sigmoid in hidden layers is the textbook cause of vanishing gradients.
# This experiment makes the convergence difference concrete on real data.
# ─────────────────────────────────────────────────────────────────────────────

class ActivationNet(nn.Module):
    """4-layer network with configurable hidden activation."""
    def __init__(self, n_in, act_fn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), act_fn(),
            nn.Linear(128,   64), act_fn(),
            nn.Linear(64,    32), act_fn(),
            nn.Linear(32,     1)
        )
    def forward(self, x): return self.net(x)


act_results = {}
for act_name, act_fn in [('Sigmoid (hidden layers)', nn.Sigmoid),
                          ('ReLU    (hidden layers)', nn.ReLU)]:
    torch.manual_seed(42)
    model = ActivationNet(n_features, act_fn).to(device)
    _, vl = run_training(model, train_loader, val_loader, criterion,
                         optim.Adam(model.parameters(), lr=1e-3),
                         num_epochs=40, device=device, verbose_every=None)
    act_results[act_name] = vl
    print(f'{act_name}  →  final val loss: {vl[-1]:.4f}')

print()
print('ReLU converges faster and reaches a lower final loss because:')
print('  1. Gradient is exactly 1 for positive inputs — no vanishing')
print('  2. He init is matched to ReLU — weights start well-scaled')
print('  3. Computationally cheaper — just a max(0, x) operation')

---
# 3.7 ModelPipeline: From Training to Deployment

## Why saving weights alone is not enough

A trained model is only useful if it can be reliably run later — in a different session, on a different machine, by a different person. Weights alone are not enough. To reproduce a prediction exactly on new data, you need everything that transformed that data before the model ever saw it:

| What you need | Why |
|--------------|-----|
| **Model architecture config** | To reconstruct the exact same network structure before loading weights |
| **Weights (`state_dict`)** | The learned parameters — what training actually produced |
| **Preprocessor** | The fitted scaler — must transform new data identically to training data |
| **Feature names** | To verify that incoming data has the right columns in the right order |
| **Training history** | Loss curves — useful for diagnosing training and continuing it |

The `ModelPipeline` class bundles all of this into a single `.pth` file.

## Why `state_dict` and not `torch.save(model)`

`torch.save(model)` uses Python's `pickle` to serialise the entire model object. Pickle stores a *reference* to the Python class — not the class itself. If you load that file anywhere the class definition is not present, loading fails. This means every deployment machine needs a copy of your training code. That is fragile.

Our `ModelPipeline` solves this properly by saving the **model class source code** as a string, alongside the weights:

| What is saved | How | Effect at load time |
|--------------|-----|--------------------|
| **Weights** | `model.state_dict()` — plain dict of tensor names → tensor values | Loads as pure data, no class dependency |
| **Model class** | `dill.dumps(model_class)` — serialises the class itself, including methods | `dill.loads()` restores the class in memory — works for notebook-defined classes |
| **Model config** | Plain Python dict, e.g. `{'n_in': 8, 'hidden_1': 128, ...}` | Passed as kwargs to the restored class to build the architecture |
| **Model class name** | String, e.g. `'CaliforniaNet'` | Display only — shown in the load summary |

**The complete load workflow — no external files needed:**
```
1. Load the checkpoint dict from the .pth file
2. dill.loads(checkpoint['model_class_bytes'])  → class is restored in memory
3. model = ModelClass(**checkpoint['model_config'])  → architecture built
4. model.load_state_dict(checkpoint['state_dict'])   → weights loaded
```

> **Why `dill` and not `inspect.getsource`?**  
> `inspect.getsource()` reads the source code from the `.py` file the class was defined in — it does not work for classes defined in Jupyter notebook cells, which have no file on disk. `dill` serialises the class object itself (methods, closures, everything) directly into bytes. This works in both notebooks and `.py` files, and is the correct production approach.

The `.pth` file is now **completely self-contained** — weights, architecture, preprocessor, and metadata all in one place. You can copy it to any machine that has PyTorch, scikit-learn, and dill installed and load it with a single call, no training code required.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7a — The ModelPipeline class
#
# This is used in every chapter from here on.
# Read through each method — all five are exercised in the cells that follow.
# ─────────────────────────────────────────────────────────────────────────────

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Weights stored as state_dict; model class stored as source code string.
    """
    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model             = model
        self.model_config      = model_config          # dict of constructor kwargs
        self.preprocessor      = preprocessor          # fitted scaler / encoder
        self.feature_names     = feature_names or []
        self.feature_ranges    = feature_ranges or {}  # {col_name: (min, max)}
        self._model_class      = model_class           # class object — kept for source extraction
        self.model_class       = model_class.__name__ if model_class else type(model).__name__
        self.training_history  = {'train_loss': [], 'val_loss': []}

    # ── Save ──────────────────────────────────────────────────────────────────
    def save(self, path):
        """Save the complete pipeline to a single .pth file.

        The model class is serialised with dill so it can be reconstructed
        on any machine without the original training code.
        dill handles classes defined in Jupyter notebooks; inspect.getsource
        only works for classes defined in .py files.
        """
        import dill
        cls_obj = self._model_class if self._model_class is not None else type(self.model)
        # dill.dumps serialises the class itself (not just its name),
        # so it can be reconstructed with dill.loads on any machine.
        class_bytes = dill.dumps(cls_obj)

        checkpoint = {
            'model_class'       : self.model_class,         # class name string — for display
            'model_class_bytes' : class_bytes,              # dill-serialised class — for reconstruction
            'model_config'      : self.model_config,        # constructor kwargs
            'state_dict'        : self.model.state_dict(),  # weights — no class dependency
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved → {path}')
        print(f'  model_class   : {self.model_class}')
        print(f'  features      : {len(self.feature_names)}')
        print(f'  saved_at      : {checkpoint["saved_at"]}')

    # ── Load ──────────────────────────────────────────────────────────────────
    @classmethod
    def load(cls, path, device=None):
        """Load a saved pipeline. No training code or class definition needed.

        The model class is reconstructed from the dill bytes saved inside
        the .pth file. The weights are then loaded into the reconstructed
        architecture. Works for classes defined in .py files and notebooks.

        Args:
            path   : path to the .pth file
            device : torch.device — defaults to CPU if not specified
        """
        import dill
        device     = device or torch.device('cpu')
        checkpoint = torch.load(path, map_location=device, weights_only=False)

        # Step 1: reconstruct the model class from dill bytes
        #   dill.loads reverses dill.dumps — the class is fully restored
        #   in memory, including its methods and any closures.
        model_class = dill.loads(checkpoint['model_class_bytes'])

        # Step 2: build the architecture from saved config kwargs
        model = model_class(**checkpoint['model_config'])
        # Step 3: load the learned weights into the reconstructed architecture
        model.load_state_dict(checkpoint['state_dict'])
        model.to(device)
        model.eval()   # always start in eval mode — correct for inference

        pipeline = cls(
            model          = model,
            model_config   = checkpoint['model_config'],
            preprocessor   = checkpoint.get('preprocessor'),
            feature_names  = checkpoint.get('feature_names', []),
            feature_ranges = checkpoint.get('feature_ranges', {}),
            model_class    = model_class,
        )
        pipeline.training_history = checkpoint.get(
            'training_history', {'train_loss': [], 'val_loss': []})

        print(f'Pipeline loaded ← {path}')
        print(f'  model_class    : {checkpoint.get("model_class")}  (reconstructed from saved class)')
        print(f'  saved_at       : {checkpoint.get("saved_at")}')
        print(f'  pytorch_version: {checkpoint.get("pytorch_version")}')
        return pipeline

    # ── Validate input ─────────────────────────────────────────────────────────
    def validate_input(self, X):
        """Check incoming data against the schema recorded at training time.

        Accepts a NumPy array or a pandas DataFrame.
        Both paths check: shape, nulls/NaNs, and value ranges.
        DataFrame additionally checks column names.
        Raises ValueError with a clear, actionable message on failure.
        """
        if isinstance(X, pd.DataFrame):
            # ── Column name checks (DataFrame only) ───────────────────────
            missing = set(self.feature_names) - set(X.columns)
            if missing:
                raise ValueError(f'Missing columns: {sorted(missing)}')
            extra = set(X.columns) - set(self.feature_names)
            if extra:
                raise ValueError(f'Unexpected extra columns: {sorted(extra)}')
            # ── Null check ────────────────────────────────────────────────
            null_cols = [c for c in self.feature_names if X[c].isnull().any()]
            if null_cols:
                raise ValueError(f'Null values in columns: {null_cols}')
            # ── Range check ───────────────────────────────────────────────
            for col, (lo, hi) in self.feature_ranges.items():
                if col in X.columns:
                    if X[col].min() < lo or X[col].max() > hi:
                        raise ValueError(
                            f'Column "{col}" out of expected range [{lo:.3f}, {hi:.3f}]')
        else:  # NumPy array
            # ── Shape check ───────────────────────────────────────────────
            if X.ndim != 2:
                raise ValueError(f'Expected a 2-D array, got shape {X.shape}')
            if self.feature_names and X.shape[1] != len(self.feature_names):
                raise ValueError(
                    f'Expected {len(self.feature_names)} features, got {X.shape[1]}')
            # ── Null / NaN check ──────────────────────────────────────────
            nan_cols = np.where(np.isnan(X).any(axis=0))[0].tolist()
            if nan_cols:
                names = [self.feature_names[i] for i in nan_cols] if self.feature_names else nan_cols
                raise ValueError(f'NaN values in columns: {names}')
            # ── Range check ───────────────────────────────────────────────
            if self.feature_names and self.feature_ranges:
                for i, name in enumerate(self.feature_names):
                    if name in self.feature_ranges:
                        lo, hi = self.feature_ranges[name]
                        col_min = float(X[:, i].min())
                        col_max = float(X[:, i].max())
                        if col_min < lo or col_max > hi:
                            raise ValueError(
                                f'Column "{name}" (index {i}) out of expected range '
                                f'[{lo:.3f}, {hi:.3f}] — got [{col_min:.3f}, {col_max:.3f}]')
        return True

    # ── Predict ───────────────────────────────────────────────────────────────
    def predict(self, X, device=None):
        """Validate input, apply preprocessor, run inference.

        Args:
            X      : raw (unscaled) NumPy array or pandas DataFrame
            device : torch.device — defaults to CPU
        Returns:
            predictions as a 1-D NumPy array, shape (n_samples,)
        """
        device = device or torch.device('cpu')
        self.validate_input(X)
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].to_numpy()
        if self.preprocessor is not None:
            X = self.preprocessor.transform(X)
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        self.model.eval()
        with torch.no_grad():
            return self.model(X_t).squeeze().cpu().numpy()

    # ── Retrain ───────────────────────────────────────────────────────────────
    def retrain(self, train_loader, val_loader, criterion, optimiser,
                num_epochs, device=None, patience=None):
        """Continue training from current weights. Appends to training_history.

        Args:
            patience : optional early stopping patience (epochs without improvement)
        """
        device = device or torch.device('cpu')
        es     = EarlyStopping(patience=patience) if patience else None
        for epoch in range(num_epochs):
            tl = train_one_epoch(self.model, train_loader, criterion, optimiser, device)
            vl = evaluate(self.model, val_loader, criterion, device)
            self.training_history['train_loss'].append(tl)
            self.training_history['val_loss'].append(vl)
            if epoch % 5 == 0:
                print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
            if es and es.step(vl, self.model):
                print(f'  Early stopping at epoch {epoch}  (best val: {es.best_loss:.4f})')
                es.restore_best(self.model)
                break


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7b — Define the model class with config-compatible constructor, then train
#
# The constructor must accept keyword arguments that match model_config exactly.
# On load, the pipeline reconstructs the class from its saved source code and
# calls: CaliforniaNet(**checkpoint['model_config'])
# The class definition — and therefore the model architecture — travels
# inside the .pth file. Nothing extra needs to be copy-pasted.
# ─────────────────────────────────────────────────────────────────────────────

class CaliforniaNet(nn.Module):
    """Regularised feedforward network for the California Housing regression task."""
    def __init__(self, n_in, hidden_1=128, hidden_2=64, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden_1),
            nn.BatchNorm1d(hidden_1), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(hidden_1, hidden_2),
            nn.BatchNorm1d(hidden_2), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(hidden_2, 1)
        )
    def forward(self, x): return self.net(x)


model_config = {
    'n_in'      : n_features,
    'hidden_1'  : 128,
    'hidden_2'  : 64,
    'dropout_p' : 0.3,
}

torch.manual_seed(42)
final_model = CaliforniaNet(**model_config).to(device)
final_opt   = optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
final_es    = EarlyStopping(patience=10)

print('Training CaliforniaNet (up to 150 epochs with early stopping):')
print(f"{'Epoch':>6}  {'Train':>8}  {'Val':>8}")
print('-' * 28)

train_hist, val_hist = [], []
for epoch in range(150):
    tl = train_one_epoch(final_model, train_loader, criterion, final_opt, device)
    vl = evaluate(final_model, val_loader, criterion, device)
    train_hist.append(tl)
    val_hist.append(vl)
    if epoch % 20 == 0:
        print(f'{epoch:>6}  {tl:>8.4f}  {vl:>8.4f}')
    if final_es.step(vl, final_model):
        print(f'\nEarly stopping at epoch {epoch}  '
              f'(best val loss: {final_es.best_loss:.4f})')
        final_es.restore_best(final_model)
        break

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7c — Wrap in ModelPipeline and save
# ─────────────────────────────────────────────────────────────────────────────

# Feature ranges must come from the RAW (unscaled) training data.
# validate_input() is called on raw features before the preprocessor is applied,
# so ranges stored as scaled values would reject all valid real-world inputs.
X_train_raw = scaler.inverse_transform(X_train)   # undo scaling to get original values
feature_ranges = {
    name: (float(X_train_raw[:, i].min()), float(X_train_raw[:, i].max()))
    for i, name in enumerate(feature_names)
}

print('Feature ranges (raw values — used for input validation at inference):')
for name, (lo, hi) in feature_ranges.items():
    print(f'  {name:12s}: {lo:8.3f} to {hi:8.3f}')
print()

pipeline = ModelPipeline(
    model          = final_model,
    model_config   = model_config,
    preprocessor   = scaler,           # fitted StandardScaler
    feature_names  = feature_names,    # list of 8 feature names
    feature_ranges = feature_ranges,   # raw-value ranges for validation
    model_class    = CaliforniaNet,
)
pipeline.training_history = {'train_loss': train_hist, 'val_loss': val_hist}

save_path = '/content/california_pipeline.pth'
pipeline.save(save_path)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7d — Load in a completely clean session and predict on the test set
#
# Notice: no model_class argument — the class is reconstructed from the
# source code saved inside the .pth file. This works on any machine.
# ─────────────────────────────────────────────────────────────────────────────

loaded = ModelPipeline.load(save_path, device=device)
print()

# predict() expects raw (unscaled) features — it applies the scaler internally
X_test_raw = scaler.inverse_transform(X_test)
preds_log  = loaded.predict(X_test_raw, device=device)

# Convert from log scale back to dollar values
preds_usd  = np.expm1(preds_log)  * 100_000
actual_usd = np.expm1(y_test)     * 100_000

rmse = np.sqrt(np.mean((preds_usd - actual_usd) ** 2))
mae  = np.mean(np.abs(preds_usd - actual_usd))

print(f'Test set performance:')
print(f'  RMSE : ${rmse:>10,.0f}')
print(f'  MAE  : ${mae:>10,.0f}')
print()
print(f'{"Predicted":>15}  {"Actual":>15}  {"Error":>14}')
print('-' * 48)
for p, a in zip(preds_usd[:8], actual_usd[:8]):
    print(f'  ${p:>12,.0f}  ${a:>12,.0f}  ${p-a:>+12,.0f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7e — Retrain on new data and save updated pipeline
# ─────────────────────────────────────────────────────────────────────────────

# Simulate a new data batch — use val set as stand-in
new_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)

print('Retraining on new data (15 additional epochs):')
retrain_opt = optim.Adam(loaded.model.parameters(), lr=5e-4, weight_decay=1e-4)

loaded.retrain(
    train_loader = new_loader,
    val_loader   = val_loader,
    criterion    = criterion,
    optimiser    = retrain_opt,
    num_epochs   = 15,
    device       = device,
)

v2_path = '/content/california_pipeline_v2.pth'
loaded.save(v2_path)
print(f'\nUpdated pipeline saved → {v2_path}')
print(f'Training history: {len(loaded.training_history["val_loss"])} total epochs recorded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.7f — Input validation: catching bad data before it reaches the model
# ─────────────────────────────────────────────────────────────────────────────

print('validate_input() — six test cases (NumPy and DataFrame paths):\n')

# ── NumPy path ────────────────────────────────────────────────────────────────

print('--- NumPy array path ---')
print()

# Case 1: wrong number of features
print('Case 1 — NumPy: wrong feature count (3 instead of 8):')
try:
    loaded.validate_input(np.random.randn(5, 3))
except ValueError as e:
    print(f'  ✓ Caught: {e}')

print()

# Case 2: NumPy array containing NaN
print('Case 2 — NumPy: array containing NaN:')
nan_arr = scaler.inverse_transform(X_test[:4]).copy()
nan_arr[1, 2] = np.nan   # inject NaN into column index 2
try:
    loaded.validate_input(nan_arr)
except ValueError as e:
    print(f'  ✓ Caught: {e}')

print()

# Case 3: NumPy array with out-of-range values
print('Case 3 — NumPy: values outside training range:')
oor_arr = scaler.inverse_transform(X_test[:4]).copy()
oor_arr[:, 0] = 999.0    # MedInc wildly out of range
try:
    loaded.validate_input(oor_arr)
except ValueError as e:
    print(f'  ✓ Caught: {e}')

print()

# ── DataFrame path ────────────────────────────────────────────────────────────

print('--- pandas DataFrame path ---')
print()

# Case 4: DataFrame missing a required column
print(f'Case 4 — DataFrame: missing required column "{feature_names[0]}":')
bad_df = pd.DataFrame(scaler.inverse_transform(X_test[:4]), columns=feature_names)
bad_df = bad_df.drop(columns=[feature_names[0]])
try:
    loaded.validate_input(bad_df)
except ValueError as e:
    print(f'  ✓ Caught: {e}')

print()

# Case 5: DataFrame with null values
print('Case 5 — DataFrame: null value in a column:')
null_df = pd.DataFrame(scaler.inverse_transform(X_test[:4]), columns=feature_names)
null_df.iloc[1, 2] = np.nan
try:
    loaded.validate_input(null_df)
except ValueError as e:
    print(f'  ✓ Caught: {e}')

print()

# Case 6: valid input (both paths) — all checks pass
print('Case 6 — Valid input (all checks pass):')
good_arr = scaler.inverse_transform(X_test[:4])
good_df  = pd.DataFrame(good_arr, columns=feature_names)
print(f'  NumPy  : {loaded.validate_input(good_arr)}  ✓')
print(f'  DataFrame: {loaded.validate_input(good_df)}  ✓')

---
## Chapter 3 Summary

| Concept | Key takeaway |
|---------|-------------|
| **Gradient descent** | Walk opposite to the gradient; steps shrink automatically as the gradient approaches zero |
| **Mini-batch GD** | Always used — one update per batch; balances speed, stability, and GPU efficiency |
| **SGD → Momentum** | Velocity accumulation eliminates ravine oscillation and builds speed in consistent directions |
| **Momentum → RMSprop** | Per-parameter adaptive LR handles features of very different scales |
| **RMSprop → Adam** | Combines both mechanisms; bias correction stabilises early steps; best general default |
| **Learning rate** | The most impactful hyperparameter — too large diverges, too small crawls, good one converges smoothly |
| **LR scheduling** | Decay over time: explore quickly early, settle precisely late |
| **Vanishing gradients** | Use ReLU and He init; avoid Sigmoid in hidden layers of deep networks |
| **Exploding gradients** | Use `clip_grad_norm_`; principled init reduces severity |
| **He initialisation** | PyTorch default for `nn.Linear`; accounts for ReLU zeroing half its inputs |
| **Dropout** | Zeros random activations during training; forces redundant, robust representations |
| **Batch Normalisation** | Normalises per mini-batch; stabilises training; correct order: Linear → BN → Act → Dropout |
| **Early stopping** | Stop when val loss plateaus; restore best weights — principled alternative to fixed epochs |
| **ModelPipeline** | Bundles model source + config + weights + preprocessor + metadata into one `.pth` file |
| **Self-contained `.pth` file** | `inspect.getsource()` saves the class definition; `exec()` on load — no training code needed on production machines |

---
## What Comes Next

Chapter 4 puts everything from this chapter into practice on two real business problems using **Feedforward Neural Networks**:

- **MNIST** — handwritten digit recognition; establishes the complete FFN classification pattern with `CrossEntropyLoss` and `Softmax`
- **Telco Customer Churn** — predicting which customers will leave; a full tabular classification pipeline from raw data to saved model
- **Counting parameters** — understanding model capacity and the trade-off between network depth and width
- **ModelPipeline: FFN** — saving and loading the churn classifier using the `ModelPipeline` introduced in this chapter

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*

---
## Answer Key — Chapter 3

---

### Exercise 3.1 — Implement Gradient Descent

```python
for lr in [0.05, 0.5, 1.1]:
    w = 5.0
    for _ in range(20):
        w = w - lr * grad_f_ex(w)
    print(f'lr={lr}: final w={w:.4f}  loss={f_ex(w):.6f}')
```

Expected results and observations:
- `lr=0.05` → w ≈ −1.64 (not yet at −2 after 20 steps). Converges eventually but too slowly. Needs more steps.
- `lr=0.5`  → w ≈ −2.000 (converged). This learning rate finds the minimum cleanly.
- `lr=1.1`  → w diverges — each step overshoots the minimum and the error *grows*. Loss increases without bound.

---

### Exercise 3.2 — Understand the Evolution

**Task 1:** Momentum solves the **ravine oscillation** problem. In ravine-shaped loss surfaces, the gradient is large and alternating perpendicular to the ravine floor, and small but consistent along the floor. SGD oscillates back and forth across the steep walls instead of advancing along the gentle floor. Momentum accumulates velocity in consistent gradient directions and cancels it in alternating directions, damping the oscillations and accelerating progress toward the minimum.

**Task 2:** For a parameter with consistently large gradients, $s_t$ (the running mean of squared gradients) becomes large. The effective step size $\eta / \sqrt{s_t}$ therefore shrinks. This is desirable because large gradients often indicate a steep dimension of the loss surface where full-size steps would overshoot. Automatically shrinking the step for that parameter keeps updates stable without requiring manual tuning of the learning rate per parameter.

```python
# Task 3
torch.manual_seed(42)
model = BaselineNet(n_features).to(device)
opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
_, vl = run_training(model, train_loader, val_loader, criterion, opt,
                     num_epochs=40, device=device, verbose_every=None)
print(f'AdamW final val loss: {vl[-1]:.4f}')
# AdamW decouples weight decay from the adaptive gradient update, which is
# mathematically more correct than Adam + weight_decay. Often performs slightly
# better, especially with larger weight_decay values.
```

---

### Exercise 3.3 — Learning Rate Scheduling

```python
torch.manual_seed(42)
model = BaselineNet(n_features).to(device)
opt   = optim.Adam(model.parameters(), lr=1e-3)
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

for epoch in range(1, 61):
    train_one_epoch(model, train_loader, criterion, opt, device)
    vl = evaluate(model, val_loader, criterion, device)
    sched.step(vl)   # must receive val_loss — not just sched.step()
    if epoch in [1, 20, 40, 60]:
        print(f'Epoch {epoch:2d}: lr={opt.param_groups[0]["lr"]:.2e}  val={vl:.4f}')
```

On California Housing, CosineAnnealingLR often matches or slightly outperforms ReduceLROnPlateau because its smooth pre-planned decay avoids abrupt drops. ReduceLROnPlateau is more adaptive but can reduce the LR prematurely or too conservatively depending on the patience setting.

---

### Exercise 3.4 — Weight Initialisation

```python
def apply_xavier_init(model):
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model

torch.manual_seed(42)
model = apply_xavier_init(DeepNet(n_features)).to(device)
opt   = optim.Adam(model.parameters(), lr=1e-3)
_, vl = run_training(model, train_loader, val_loader, criterion, opt,
                     num_epochs=30, device=device, verbose_every=None)
print(f'Xavier init final val loss: {vl[-1]:.4f}')
```

**Task 2:** Xavier often still converges on shallower networks, but He init is preferred because it is *derived specifically for ReLU*. ReLU zeroes approximately half its inputs on any given batch, effectively halving the signal variance passing through each layer. He init compensates with a $\sqrt{2/n_{in}}$ factor — twice the scale of Xavier. As network depth increases, this difference compounds: Xavier-initialised deep ReLU networks converge more slowly and are more sensitive to learning rate choice.

---

### Exercise 3.5 — Regularisation

```python
for p in [0.1, 0.3, 0.5]:
    torch.manual_seed(42)
    model = RegularisedNet(n_features, dropout_p=p).to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    tl_h, vl_h = run_training(model, train_loader, val_loader, criterion, opt,
                               num_epochs=60, device=device, verbose_every=None)
    tl_f = evaluate(model, train_loader, criterion, device)
    print(f'p={p}: train={tl_f:.4f}  val={vl_h[-1]:.4f}  gap={vl_h[-1]-tl_f:.4f}')
```

**Task 2:** During training, Dropout randomly disables a fraction of neurons at each forward pass — the model is effectively a different, smaller network at every step. This makes the training task harder, so training loss is elevated. At evaluation, `model.eval()` turns Dropout off — all neurons contribute — and the model performs better than its noisy training self suggests. A higher training loss alongside a lower validation loss is therefore a sign that Dropout is working correctly. It is not overfitting in the usual direction — it is *preventing* it.